In [0]:
%run "/Workspace/Repos/sjdgustavo@gmail.com/databricks-etl-pos/ETL_arquitetura_medalhao/00.config/config"

In [0]:
from pyspark.sql import functions as F  
from pyspark.sql.functions import to_date, regexp_replace, col, greatest

In [0]:
ipca = spark.table(f"{CATALOG}.{BRONZE}.ipca")
boi = spark.table(f"{CATALOG}.{BRONZE}.boi_gordo")

print("Schema IPCA:")
ipca.printSchema()

print("Schema BOI:")
boi.printSchema()


In [0]:
from pyspark.sql.functions import col 

boi = (
    boi
    .withColumnRenamed("Data", "data")
    .withColumnRenamed("Valor", "boi_gordo")
)

display(boi)
boi.printSchema()

In [0]:
from pyspark.sql.functions import to_date, date_format

boi = boi.withColumn("data", to_date("data", "MM/yyyy"))

boi = boi.withColumn("data", date_format("data", "yyyy-MM"))

display(boi)

In [0]:
ipca = ipca.withColumn("data", to_date("data", "dd/MM/yyyy"))
ipca = ipca.withColumn("data", date_format("data", "yyyy-MM"))

display (ipca)

In [0]:
df_join = (
    ipca.join(boi, on="data", how="inner")
        .select("data", "ipca", "boi_gordo")
)

display(df_join)

In [0]:
ip = ipca.alias("ip")
bo = boi.alias("bo")

df_join = (
    ip.join(bo, col("ip.data") == col("bo.data"), "inner")
        .select(
            col("ip.data").alias("data"),
            col("ip.ipca").alias("ipca"),
            col("bo.boi_gordo").alias("boi_gordo"),
            col("ip.data_coleta").alias("data_coleta")
        )
)

display(df_join)

In [0]:
from pyspark.sql.functions import regexp_replace

df_join = df_join.withColumn("data", to_date(col("data"), "yyyy-MM"))

df_join = (
    df_join
    .withColumn("ipca", regexp_replace("ipca", ",", ".").cast("double"))
    .withColumn("boi_gordo", regexp_replace("boi_gordo", ",", ".").cast("double"))
    
)

df_join.printSchema()
display(df_join.orderBy("data").limit(50))

In [0]:
df_join.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SILVER}.economia")